In [1]:
#TODO make sure this renders in the github repo

# Decoder

![decoder](../showcase_images/decoder.png)

- Composed of a stack of $Nx$ identical decoder layers.
- The data flow goes like this: 
  1. Input is passed through the first decoder layer.
  2. The output of the first decoder then feeds into the second decoder layer, and so on. 
  3. Once it reaches the last decoder layer, it will feed the RMSNorm that is outside of the decoder.

In [2]:
import easyjupyter
import torch
import torch.nn as nn
from typing import Optional
from model.SwiGLU_FFN import SwiGLUFeedForward
from model.GQA import GroupedQueryAttention
from model.RMSNorm import RMSNorm

In [3]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_configs import BaseConfig

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        """
        A single decoder layer of the Llama Decoder Nx stack.
        """
        super().__init__()

        self.self_attn = GroupedQueryAttention(cfg)  # Handle GQA and RoPE
        self.ffn = SwiGLUFeedForward(cfg)

        # Pre-normalization: Must have separate RMSNorm layers, so each can have a different learned weights
        self.attention_norm = RMSNorm(cfg)
        self.ffn_norm = RMSNorm(cfg)

    def forward(
        self,
        x: torch.Tensor,
        freqs_cis: torch.Tensor,
        combined_mask: Optional[torch.Tensor] = None,
        kv_cache: Optional[tuple] = None,
    ) -> tuple[torch.Tensor, Optional[tuple]]:
        """
        Args:
            x: The input tensor of shape (batch_size, seq_len, hidden_size).
            freqs_cis: The precomputed tensor for RoPE.
            freqs_cis: The precomputed frequencies for RoPE.
            combined_mask: This mask contains the causal mask which prevents tokens from attending to future tokens, and the document mask which prevents a token in Doc A from attending to tokens in other documents.
                - Paper mentioned: "We use an attention mask that prevents self-attention between different documents within the same sequence. We find that this change had limited impact during in standard pre-training, but find it to be important in continued pre-training on very long sequences."
            kv_cache: Inference only, Tuple of (key, value) for autoregressive inference.
        
        Returns: 
            A tuple of (output, updated_kv_cache) for this n'th decoder layer.
        """
        # TODO add the cross-attention layer
        residual_1 = x

        # 1. Apply RMS Norm
        x = self.attention_norm(x)
        # 2. Apply GQA with RoPE
        x, updated_kv_cache = self.self_attn(x, freqs_cis, combined_mask, kv_cache)
        # 3. Add first residual connection
        x = residual_1 + x

        residual_2 = x

        # 4 Apply second RMS Norm
        x = self.ffn_norm(x)
        # 5 Apply Feed Forward SwiGLU
        x = self.ffn(x)
        # 6 Add second residual connection
        x = residual_2 + x

        return x, updated_kv_cache

In [5]:
class Decoder(nn.Module):
    def __init__(self, cfg: BaseConfig):
        """
        The full Decoder that is a Nx stack of DecoderLayers().
        """
        super().__init__()

        # Make the Nx decoder layers
        self.layers = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.num_layers)])

    def forward(
        self,
        x: torch.Tensor,
        freqs_cis: torch.Tensor,
        combined_mask: Optional[torch.Tensor] = None,
        kv_caches: Optional[list[tuple]] = None,
    ):
        """
        Args:
            x: The input tensor of shape (batch_size, seq_len, hidden_size).
            freqs_cis: The precomputed tensor for RoPE.
            freqs_cis: The precomputed frequencies for RoPE.
            combined_mask: This mask contains the causal mask which prevents tokens from attending to future tokens, and the document mask which prevents a token in Doc A from attending to tokens in other documents.
                - Paper mentioned: "We use an attention mask that prevents self-attention between different documents within the same sequence. We find that this change had limited impact during in standard pre-training, but find it to be important in continued pre-training on very long sequences."
            kv_caches: Inference only, a list of kv caches.
        """
        updated_kv_caches = []

        # Iterate through the decoder layers, passing the corresponding cache to each.
        for i, layer in enumerate(self.layers):
            layer_cache = kv_caches[i] if kv_caches is not None else None

            # Forward pass through the layer
            x, updated_layer_kv_cache = layer(x, freqs_cis, combined_mask, layer_cache)

            # Store the updated cache
            if updated_layer_kv_cache is not None:
                updated_kv_caches.append(updated_layer_kv_cache)

        # Return the final tensor and fully updated list of kv caches
        return x, (updated_kv_caches if len(updated_kv_caches) < 0 else None)

### Test

In [9]:
# @i-c
from llama_configs import Llama3_scaled_down
from model.bpe_tokenizer import BPETokenizer

cfg = Llama3_scaled_down()
cfg.max_docs = cfg.overfit_max_docs
cfg.vocab_size = 32_000
cfg.token_budget = 250 * cfg.global_batch_size_tokens

decoder = Decoder(cfg).to(cfg.device)

t = BPETokenizer(cfg)
success, loaded_tokenizer = t.load_tokenizer()
if not success:
    raise ValueError(
        f"Tokenizer file not found!\n"
        "\n\nPlease run `python prepare.py --d pretrain --overfit` first."
    )

text_str = "The brown rabbit ate the apple"
encoded = loaded_tokenizer.encode(text_str)
print(f"Original string: {text_str}")
print(f"Token IDs: {encoded.ids}")
print(f"Tokens: {encoded.tokens}")


Project Root: /Users/tonyavis/Main/Build_an_LLM

Loading existing trained BPE Tokenizer with vocab size: 32000 (Trained on 1000 documents) from: (/Users/tonyavis/Main/Build_an_LLM/artifacts/universal_tokenizer/bpe_vocab_size_32000_samples_1000.json)...
Original string: The brown rabbit ate the apple
Token IDs: [1, 449, 5690, 7011, 1585, 12317, 265, 9782]
Tokens: ['<|begin_of_doc|>', 'The', 'Ġbrown', 'Ġrab', 'bit', 'Ġate', 'Ġthe', 'Ġapple']


In [10]:
# @i-c
# Masks
from model.utils.masking import create_causal_doc_mask

token_ids_tensor = torch.tensor(encoded.ids, dtype=torch.long, device=cfg.device).unsqueeze(0)
combined_mask = create_causal_doc_mask(cfg, token_ids_tensor)

head_dim = cfg.d_model // cfg.attn_heads

token_embeds = nn.Embedding(
            num_embeddings=cfg.vocab_size, embedding_dim=cfg.d_model
        ).to(cfg.device)

x_embeds = token_embeds(token_ids_tensor)

from model.RoPE import precompute_freqs_cis
freqs_cis = precompute_freqs_cis(cfg, head_dim=head_dim)

# Patch fix: because dummy tokens is only 6 tokens long == ("The brown rabbit ate the apple")
seq_len = x_embeds.shape[1]
freqs_cis = freqs_cis[:seq_len]

# Decoder forward pass
print("\n\n---Testing Decoder Forward Pass:")
decoder_out, kv_caches = decoder(x_embeds, freqs_cis, combined_mask)
print("kv_caches",kv_caches)
print(
    f"\nInput shape: {x_embeds.shape}\n"
    f"Decoder output shape: {decoder_out.shape}"
    # f"Number of KV caches generated: {len(kv_caches)}" # Is None when not in inference mode
)



---Testing Decoder Forward Pass:
kv_caches None

Input shape: torch.Size([1, 8, 768])
Decoder output shape: torch.Size([1, 8, 768])
